# A3.1 SVM y multiple testing
#### Camila Johana González Acosta 599303

El objetivo es realizar análisis de inferencia (2 vs 4 y multiclase) con correcciones por pruebas múltiples,
y entrenar clasificadores SVM con distintos kernels. 

In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 10
pd.options.display.max_rows = 20


## 1) Cargar datos y revisar huecos

In [19]:
path = 'A3.1 Khan.csv'  
df = pd.read_csv(path, index_col=None)
print('Dimensiones:', df.shape)
display(df.head())

if 'Class' in df.columns:
    label_col = 'Class'
else:
    label_col = df.columns[-1]
print('Columna de etiqueta usada:', label_col)

print('\nValores faltantes por columna (sum):')
display(df.isnull().sum().sort_values(ascending=False).head(10))
print('\nTotal valores faltantes en el dataset:', df.isnull().sum().sum())

X = df.drop(columns=[label_col])
y = df[label_col].astype(int)

print('\nClases presentes y conteos:')
display(y.value_counts())

mean2 = X[y==2].mean()
mean4 = X[y==4].mean()
diff24 = (mean2 - mean4).abs().sort_values(ascending=False)
top10 = diff24.head(10)
print('\nTop 10 genes con mayor diferencia de medias (clase 2 vs 4):')
display(top10.to_frame(name='|mean2 - mean4|'))

Dimensiones: (83, 2309)


,X1,X2,X3,X4,X5,...,X2305,X2306,X2307,X2308,y
0,0.773344,-2.438405,-0.482562,-2.721135,-1.217058,...,-5.496768,-1.414282,-0.647600,-1.763172,2
1,-0.078178,-2.415754,0.412772,-2.825146,-0.626236,...,-3.661264,-1.093923,-1.209320,-0.824395,2
2,-0.084469,-1.649739,-0.241308,-2.875286,-0.889405,...,-2.736450,-1.965399,-0.805868,-1.139434,2
3,0.965614,-2.380547,0.625297,-1.741256,-0.845366,...,-2.077843,-1.127629,0.331531,-2.179483,2
4,0.075664,-1.728785,0.852626,0.272695,-1.841370,...,-1.675044,-1.082050,-0.965218,-1.836966,2


Columna de etiqueta usada: y

Valores faltantes por columna (sum):


X1     0
X2     0
X3     0
X4     0
X5     0
X6     0
X7     0
X8     0
X9     0
X10    0
dtype: int64


Total valores faltantes en el dataset: 0

Clases presentes y conteos:


y
2    29
4    25
3    18
1    11
Name: count, dtype: int64


Top 10 genes con mayor diferencia de medias (clase 2 vs 4):


,|mean2 - mean4|
X187,3.323151
X509,2.906537
X2046,2.424515
X2050,2.401783
X129,2.165185
X1645,2.065460
X1319,2.045941
X1955,2.037340
X1003,2.011337
X246,1.837830


El dataset contiene 83 muestras y 2309 variables sin valores faltantes. Se identifican cuatro clases con diferentes cantidades de casos, siendo la clase 4 la más frecuente. Al comparar las clases 2 y 4, se observó que los genes X187, X509 y X2060 presentan las mayores diferencias de medias, lo que indica que podrían ser los más relevantes para distinguir entre ambas clases.

## 2) Pruebas t por gen (clase 2 vs clase 4) y corrección por múltiples pruebas

Calcularemos el estadístico t y p-value para cada gen usando `ttest_ind` (asumiendo varianzas iguales por simplicidad). Luego aplicaremos las correcciones de Bonferroni, Holm y Benjamini-Hochberg (FDR) usando `multipletests`.

Mostraremos qué genes resultan significativos con un control alpha = 0.05 para cada método.

In [20]:
genes = X.columns
t_stats = []
p_values = []

group2 = X[y==2]
group4 = X[y==4]

for g in genes:
    t, p = stats.ttest_ind(group2[g], group4[g], equal_var=True, nan_policy='omit')
    t_stats.append(t)
    p_values.append(p)

p_values = np.array(p_values)
t_stats = np.array(t_stats)

alpha = 0.05
methods = {'bonferroni':'bonferroni', 'holm':'holm', 'bh':'fdr_bh'}
results = {}
for name, method in methods.items():
    rej, pvals_corr, _, _ = multipletests(p_values, alpha=alpha, method=method)
    sig_genes = list(genes[rej])
    results[name] = {'rejected': rej, 'pvals_corr': pvals_corr, 'sig_genes': sig_genes}

for name in results:
    print(f"Método: {name} -> genes significativos: {len(results[name]['sig_genes'])}")


for name in results:
    print('\n---\nMétodo:', name)
    sigs = results[name]['sig_genes']
    if len(sigs)==0:
        print('No hay genes significativos con este método a α=0.05.')
    else:
        display(pd.DataFrame({
            'gene': sigs,
            'mean_diff': (mean2[sigs] - mean4[sigs]).abs(),
            'p_value_uncorrected': p_values[[list(genes).index(g) for g in sigs]]
        }).sort_values('mean_diff', ascending=False).head(50))


Método: bonferroni -> genes significativos: 74
Método: holm -> genes significativos: 74
Método: bh -> genes significativos: 297

---
Método: bonferroni


,gene,mean_diff,p_value_uncorrected
X187,X187,3.323151,2.532628e-16
X509,X509,2.906537,8.120968e-15
X2046,X2046,2.424515,3.946325e-15
X2050,X2050,2.401783,4.406986e-15
X129,X129,2.165185,4.702161e-12
...,...,...,...
X1330,X1330,1.043660,6.217541e-09
X2227,X2227,1.029116,3.305040e-06
X910,X910,1.024500,8.936965e-09
X2115,X2115,1.007754,3.865694e-06



---
Método: holm


,gene,mean_diff,p_value_uncorrected
X187,X187,3.323151,2.532628e-16
X509,X509,2.906537,8.120968e-15
X2046,X2046,2.424515,3.946325e-15
X2050,X2050,2.401783,4.406986e-15
X129,X129,2.165185,4.702161e-12
...,...,...,...
X1330,X1330,1.043660,6.217541e-09
X2227,X2227,1.029116,3.305040e-06
X910,X910,1.024500,8.936965e-09
X2115,X2115,1.007754,3.865694e-06



---
Método: bh


,gene,mean_diff,p_value_uncorrected
X187,X187,3.323151,2.532628e-16
X509,X509,2.906537,8.120968e-15
X2046,X2046,2.424515,3.946325e-15
X2050,X2050,2.401783,4.406986e-15
X129,X129,2.165185,4.702161e-12
...,...,...,...
X1896,X1896,1.044388,3.732323e-07
X1298,X1298,1.044341,1.292252e-05
X1330,X1330,1.043660,6.217541e-09
X1201,X1201,1.037358,1.213485e-04


Se realizó una prueba *t* entre las clases 2 y 4 para cada gen, ajustando los valores *p* con tres métodos de corrección por comparaciones múltiples. Los métodos Bonferroni y Holm identificaron 74 genes con diferencias significativas, mientras que el método Benjamini-Hochberg (BH) detectó 297. Esto indica que existe un número considerable de genes cuya expresión difiere de forma estadísticamente significativa entre ambas clases, y que el método BH, al ser menos conservador, permite detectar más posibles genes relevantes.


## 3) ANOVA (4 clases)

Realizamos un ANOVA por gen para comparar las medias entre las 4 clases. Usaremos `f_oneway` de `scipy.stats`. Mostraremos los genes con p-valor bajo y aplicaremos corrección Benjamini-Hochberg (FDR) para múltiples pruebas.

In [21]:
from scipy.stats import f_oneway
pvals_anova = []
f_stats = []
for g in genes:
    groups = [X[y==cls][g].values for cls in sorted(y.unique())]
    try:
        f, p = f_oneway(*groups)
    except Exception as e:
        f, p = np.nan, np.nan
    f_stats.append(f)
    pvals_anova.append(p)

pvals_anova = np.array(pvals_anova)
f_stats = np.array(f_stats)

rej_anova, pvals_anova_corr, _, _ = multipletests(pvals_anova, alpha=alpha, method='fdr_bh')
sig_genes_anova = list(genes[rej_anova])
print('Genes significativos por ANOVA con FDR (alpha=0.05):', len(sig_genes_anova))
if len(sig_genes_anova)>0:
    display(pd.DataFrame({
        'gene': sig_genes_anova,
        'f_stat': f_stats[[list(genes).index(g) for g in sig_genes_anova]],
        'p_value_uncorrected': pvals_anova[[list(genes).index(g) for g in sig_genes_anova]]
    }).sort_values('f_stat', ascending=False).head(50))
else:
    print('Ningún gen significativo con FDR a 0.05.')


Genes significativos por ANOVA con FDR (alpha=0.05): 1162


,gene,f_stat,p_value_uncorrected
990,X1955,84.364086,1.459035e-24
698,X1389,83.817537,1.772751e-24
501,X1003,77.795622,1.618988e-23
1037,X2050,69.230799,4.733702e-22
123,X246,68.414042,6.633722e-22
...,...,...,...
785,X1577,34.431405,2.461953e-14
390,X783,34.223484,2.815349e-14
117,X236,34.171445,2.911674e-14
189,X380,34.016799,3.218355e-14


Se aplicó un ANOVA para comparar la expresión génica entre las cuatro clases, ajustando los valores *p* con el método FDR (Benjamini-Hochberg). El análisis identificó **1162 genes significativos** con diferencias de expresión entre grupos. Los genes con mayores valores F, como **X1955, X1389 y X1003**, muestran las diferencias más marcadas entre las clases, lo que sugiere que podrían tener un papel importante en la distinción biológica entre ellas.

In [22]:
# Validación cruzada para elegir mejor valor de C en el kernel RBF

from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

C_values = [0.01, 0.1, 0.5, 1, 5, 10]
cv_scores = []

for C in C_values:
    svm = SVC(kernel='rbf', C=C, gamma=0.01)
    scores = cross_val_score(svm, X_train_sel, y_train, cv=5)
    cv_scores.append(scores.mean())
    print(f"C={C} -> Mean CV accuracy: {scores.mean():.3f}")

best_C = C_values[np.argmax(cv_scores)]
print(f"\n Mejor valor de C según validación cruzada: {best_C}")


C=0.01 -> Mean CV accuracy: 0.345
C=0.1 -> Mean CV accuracy: 0.638
C=0.5 -> Mean CV accuracy: 0.864
C=1 -> Mean CV accuracy: 1.000
C=5 -> Mean CV accuracy: 1.000
C=10 -> Mean CV accuracy: 0.983

 Mejor valor de C según validación cruzada: 1


Despues de realizar diferentes pruebas, se decidió buscar por medio de validación cruzada el mejor valor para C. 

## 4) Entrenamiento de SVMs (selección de variables y split)

Para ahorrar tiempo, seleccionaremos las 20 variables con mayor diferencia absoluta de medias entre clase 2 y 4 (resultado del punto 1) y usaremos esas como características. Luego separamos en entrenamiento/prueba y entrenamos 3 modelos SVM: linear, poly (grado 3) y rbf.

In [23]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print("Tamaños:", X_train.shape, X_test.shape)

mean2_train = X_train[y_train == 2].mean()
mean4_train = X_train[y_train == 4].mean()
diff_train = (mean2_train - mean4_train).abs().sort_values(ascending=False)

N = 20
top_genes_train = list(diff_train.index[:N])
print("Número de genes seleccionados:", len(top_genes_train))
print("Ejemplo de genes seleccionados:", top_genes_train[:10])

X_train_sel = X_train[top_genes_train]
X_test_sel = X_test[top_genes_train]

models = {
    'LinearSVC': LinearSVC(max_iter=10000),
    'SVC_poly3': SVC(kernel='poly', degree=3, C=1),
    'SVC_rbf': SVC(kernel='rbf', C=best_C, gamma=0.01)
}

for name, model in models.items():
    model.fit(X_train_sel, y_train)
    preds = model.predict(X_test_sel)
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("Balanced accuracy:", balanced_accuracy_score(y_test, preds))
    print("Classification report:\n", classification_report(y_test, preds, zero_division=0))
    print("Confusion matrix:\n", confusion_matrix(y_test, preds))


Tamaños: (58, 2308) (25, 2308)
Número de genes seleccionados: 20
Ejemplo de genes seleccionados: ['X187', 'X509', 'X2050', 'X2046', 'X129', 'X1645', 'X1319', 'X714', 'X1003', 'X246']

LinearSVC
Accuracy: 0.88
Balanced accuracy: 0.8722222222222222
Classification report:
               precision    recall  f1-score   support

           1       0.50      1.00      0.67         3
           2       1.00      0.89      0.94         9
           3       1.00      0.60      0.75         5
           4       1.00      1.00      1.00         8

    accuracy                           0.88        25
   macro avg       0.88      0.87      0.84        25
weighted avg       0.94      0.88      0.89        25

Confusion matrix:
 [[3 0 0 0]
 [1 8 0 0]
 [2 0 3 0]
 [0 0 0 8]]

SVC_poly3
Accuracy: 0.96
Balanced accuracy: 0.9722222222222222
Classification report:
               precision    recall  f1-score   support

           1       1.00      1.00      1.00         3
           2       1.00      0.89

Se seleccionaron los 20 genes con mayor diferencia de medias entre las clases 2 y 4 para entrenar tres modelos SVM. Los resultados muestran que el modelo **SVC con kernel RBF** obtuvo un **100 % de precisión y balance perfecto entre clases**, seguido por **SVC con kernel polinómico (96 %)** y **LinearSVC (88 %)**. Esto indica que los genes elegidos tienen un alto poder discriminante y que los modelos no lineales, especialmente el RBF, logran capturar mejor las diferencias complejas entre las clases del conjunto de datos.

## 5) Métricas y conclusiones

In [24]:
summary = []

for name, model in models.items():
    preds = model.predict(X_test_sel)  
    summary.append({
        'model': name,
        'accuracy': accuracy_score(y_test, preds),
        'balanced_accuracy': balanced_accuracy_score(y_test, preds)
    })
    
summary_df = pd.DataFrame(summary).set_index('model')
display(summary_df)


,accuracy,balanced_accuracy
model,,
LinearSVC,0.88,0.872222
SVC_poly3,0.96,0.972222
SVC_rbf,1.00,1.000000


El análisis reveló que muchos genes presentan diferencias significativas entre las clases, evidenciando una clara separación biológica. Los modelos SVM lograron altos niveles de precisión incluso usando solo 20 genes, destacando el SVC con kernel polinómico (96%) como el más equilibrado. Aunque el modelo RBF alcanzó 100% de exactitud, este resultado podría indicar sobreajuste, ya que el modelo puede estar memorizando los datos de entrenamiento más que generalizando.

Los resultados resaltan el potencial de estos genes para clasificar, pero subrayan la necesidad de validar los modelos con más datos.